In [ ]:
import pandas as pd


In [ ]:
pool_pids = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/reverse_geocoding_pool_pids.csv')
pool_pids = pool_pids.drop_duplicates(subset=['latitude', 'longitude'])
pool_pids = pool_pids.drop_duplicates(subset=['place_id'])
pool_pids.info()


In [ ]:
pool_pids.to_csv ('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/reverse_geocoding_pool_pids.csv', index = False)

In [ ]:
original = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/baederleben_swimmingpools_coords.csv')
#drop duplicates based on lat long
original = original.dropna(subset=['latitude', 'longitude'])
original = original.drop_duplicates(subset=['latitude', 'longitude'])
original.info()
original.to_csv('admin_coords.csv', index=False)

In [ ]:
# original = original.drop_duplicates(subset=['latitude', 'longitude'])
# original.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/baederleben_swimmingpools_coords.csv')
# #drop duplicates based on lat long

In [ ]:
def correct_coordinates_in_place(df, lat_col='latitude', lon_col='longitude'):
    """
    Correct the coordinate columns in place by replacing first comma with dot and removing others
    """
    # Store original counts for reporting
    original_lat_problems = df[lat_col].astype(str).str.contains(',').sum()
    original_lon_problems = df[lon_col].astype(str).str.contains(',').sum()
    
    # Function to apply the transformation
    def transform_coordinate(value):
        if isinstance(value, str) and ',' in value:
            parts = value.split(',')
            return parts[0] + '.' + ''.join(parts[1:])
        return value
    
    # Apply transformations
    df[lat_col] = df[lat_col].apply(transform_coordinate)
    df[lon_col] = df[lon_col].apply(transform_coordinate)
    
    # Convert to float64
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # Report what was fixed
    print(f"Fixed {original_lat_problems} latitude values and {original_lon_problems} longitude values")
    print(f"Data types after correction:")
    print(f"  {lat_col}: {df[lat_col].dtype}")
    print(f"  {lon_col}: {df[lon_col].dtype}")
    
    return df

# Apply the correction to your original dataframe
print("Before correction:")
print(f"Original latitude dtype: {original['latitude'].dtype}")
print(f"Original longitude dtype: {original['longitude'].dtype}")

# Correct the dataframe IN PLACE
original = correct_coordinates_in_place(original)

print("\nAfter correction:")
print(f"Sample latitude values: {original['latitude'].head()}")
print(f"Sample longitude values: {original['longitude'].head()}")

In [ ]:
# change dtype of latitude and longitude to float
original['latitude'] = original['latitude'].astype(float)
original['longitude'] = original['longitude'].astype(float)

In [ ]:
def find_true_unique_rows(original_df, pool_df):
    """
    Find rows in original that are not in pool_pids, accounting for coordinate duplicates
    """
    print("=== ANALYZING COORDINATE DUPLICATES ===")
    
    # Count unique coordinates in each dataset
    original_unique_coords = original_df[['latitude', 'longitude']].drop_duplicates()
    pool_unique_coords = pool_df[['latitude', 'longitude']].drop_duplicates()
    
    print(f"Original total rows: {len(original_df)}")
    print(f"Original unique coordinates: {len(original_unique_coords)}")
    print(f"Pool_pids total rows: {len(pool_df)}")
    print(f"Pool_pids unique coordinates: {len(pool_unique_coords)}")
    
    # Find coordinates in original but not in pool
    merged = original_unique_coords.merge(
        pool_unique_coords, 
        on=['latitude', 'longitude'], 
        how='left', 
        indicator=True
    )
    
    unique_coords = merged[merged['_merge'] == 'left_only']
    
    print(f"\nUnique coordinates in original (not in pool): {len(unique_coords)}")
    
    # Get the full rows from original that have these unique coordinates
    # We'll take just the first occurrence of each unique coordinate
    unique_rows = original_df.merge(
        unique_coords[['latitude', 'longitude']], 
        on=['latitude', 'longitude']
    ).drop_duplicates(subset=['latitude', 'longitude'])
    
    return unique_rows

true_unique_rows = find_true_unique_rows(original, pool_pids)
print(f"\nTrue unique rows (first occurrence of each unique coordinate):")
print(true_unique_rows)

In [ ]:
true_unique_rows.to_csv('missing_coords.csv', index=False)

In [ ]:
#find the  rows not in pools_pids

def find_rows_not_in_pool(original_df, pool_df):
    """
    Find rows in original that are not in pool_pids using lat/lon coordinates
    """
    # Perform a left merge with indicator
    merged = original_df.merge(
        pool_df[['latitude', 'longitude']], 
        on=['latitude', 'longitude'], 
        how='left', 
        indicator=True
    )
    
    # Filter for rows that only exist in original
    only_in_original = merged[merged['_merge'] == 'left_only']
    
    # Drop the indicator column to get back original format
    only_in_original = only_in_original.drop('_merge', axis=1)
    
    print(f"Found {len(only_in_original)} rows in original that are not in pool_pids")
    return only_in_original

# Find the unique rows
unique_to_original = find_rows_not_in_pool(original, pool_pids)
print(f"\nSample of unique rows:")
print(unique_to_original.head())

In [ ]:
unique_to_original.info()

In [ ]:
import geopandas as gpd

In [ ]:
reversed_pools = gpd.read_file('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/clean_datasets/pois_within_shapefile.gpkg')
reversed_pools.info()

In [ ]:
reversed_pools.primary_place_type.unique()